In [1]:
import pandas as pd
import numpy as np
import requests
import io
import math
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

自動化取得釣魚網站資料，預計設定執行檔，每小時執行一次

In [ ]:
def download_phishtank_data():
    # PhishTank 提供每小時更新的在線且已驗證釣魚網址 
    url = "http://data.phishtank.com/data/online-valid.csv"
    
    # 根據規定，必須提供自定義 User-Agent 以辨識應用程式 
    headers = {
        'User-Agent': 'phishtank/AI-Agent-Safety-Project'
    }
    
    try:
        print("正在從 PhishTank 下載最新數據...")
  
        response = requests.get(url, headers=headers)
        
        # 若遇到 HTTP 509 代表超出免費頻率限制 
        if response.status_code == 200:
            # 將 CSV 內容讀取進 Pandas 
            df = pd.read_csv(io.StringIO(response.text))
            print(f"成功下載！目前共有 {len(df)} 筆有效釣魚網址。")
            return df
        else:
            print(f"下載失敗，狀態碼：{response.status_code}")
            return None
    except Exception as e:
        print(f"發生錯誤：{e}")
        return None

# 執行下載
phish_df = download_phishtank_data()

# 查看前五筆數據 
if phish_df is not None:
    print(phish_df[['url', 'verified', 'online']].head())

正在從 PhishTank 下載最新數據...
成功下載！目前共有 47318 筆有效釣魚網址。
                                                 url verified online
0       http://allegrolokalnie.pl-oferta74630741.cfd      yes    yes
1   https://bellsouth-att-sign-in-b081d9.webflow.io/      yes    yes
2                         https://stratiela.com/red/      yes    yes
3     https://allegrolokalnie.pl-oferta74630741.cfd/      yes    yes
4  https://content.sixflags.com/news/director.asp...      yes    yes


In [5]:
phish_df

,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target
0,9296878,http://allegrolokalnie.pl-oferta74630741.cfd,http://www.phishtank.com/phish_detail.php?phis...,2025-12-22T14:03:21+00:00,yes,2025-12-22T14:12:09+00:00,yes,Allegro
1,9296877,https://bellsouth-att-sign-in-b081d9.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-12-22T14:01:06+00:00,yes,2025-12-22T14:12:09+00:00,yes,Other
2,9296873,https://stratiela.com/red/,http://www.phishtank.com/phish_detail.php?phis...,2025-12-22T13:52:49+00:00,yes,2025-12-22T14:02:23+00:00,yes,Other
3,9296871,https://allegrolokalnie.pl-oferta74630741.cfd/,http://www.phishtank.com/phish_detail.php?phis...,2025-12-22T13:48:40+00:00,yes,2025-12-22T13:52:03+00:00,yes,Allegro
4,9296865,https://content.sixflags.com/news/director.asp...,http://www.phishtank.com/phish_detail.php?phis...,2025-12-22T13:31:06+00:00,yes,2025-12-22T13:31:51+00:00,yes,Other
...,...,...,...,...,...,...,...,...
47313,4173961,http://webmailadmin0.myfreesites.net/,http://www.phishtank.com/phish_detail.php?phis...,2016-06-13T00:17:16+00:00,yes,2016-11-18T03:03:26+00:00,yes,Other
47314,2042606,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,http://www.phishtank.com/phish_detail.php?phis...,2013-09-30T13:24:39+00:00,yes,2013-10-01T13:33:17+00:00,yes,Other
47315,1865971,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,http://www.phishtank.com/phish_detail.php?phis...,2013-05-28T15:59:31+00:00,yes,2013-05-28T22:13:12+00:00,yes,Other
47316,1460953,http://www.habbocreditosparati.blogspot.com/,http://www.phishtank.com/phish_detail.php?phis...,2012-06-14T15:19:26+00:00,yes,2012-06-14T15:29:11+00:00,yes,Sulake Corporation


In [6]:
from tranco import Tranco

# 初始化 (cache 預設會快取下載的列表)
t = Tranco(cache=True, cache_dir='.tranco')

# 取得最新一天的列表
latest_list = t.list()

# 查詢特定網域的排名
rank = latest_list.rank('google.com')
print(f"Google's rank: {rank}")

# 取得前 1000 名的列表
top_1000 = latest_list.top(1000)

Google's rank: 1


In [7]:
top_1000

['google.com',
 'gtld-servers.net',
 'microsoft.com',
 'facebook.com',
 'cloudflare.com',
 'googleapis.com',
 'mail.ru',
 'youtube.com',
 'apple.com',
 'amazonaws.com',
 'instagram.com',
 'dzen.ru',
 'gstatic.com',
 'office.com',
 'twitter.com',
 'akamai.net',
 'live.com',
 'linkedin.com',
 'googlevideo.com',
 'akamaiedge.net',
 'googletagmanager.com',
 'amazon.com',
 'akadns.net',
 'fbcdn.net',
 'workers.dev',
 'microsoftonline.com',
 'windowsupdate.com',
 'azure.com',
 'wikipedia.org',
 'doubleclick.net',
 'github.com',
 'googleusercontent.com',
 'bing.com',
 'netflix.com',
 'domaincontrol.com',
 'ax-msedge.net',
 'digicert.com',
 'fastly.net',
 'e2ro.com',
 'whatsapp.net',
 't-msedge.net',
 'wordpress.org',
 'apple-dns.net',
 'aaplimg.com',
 'office.net',
 'youtu.be',
 'googlesyndication.com',
 'pinterest.com',
 'akam.net',
 'appsflyersdk.com',
 'icloud.com',
 'yahoo.com',
 'sharepoint.com',
 'trafficmanager.net',
 'ripn.net',
 'windows.net',
 'whatsapp.com',
 'goo.gl',
 'cloudfront